In [1]:
from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import logging_utils
print(logging_utils.__file__)

/workspaces/buidling_AI_agents/aihero/project/logging_utils.py


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from pydantic_ai import Agent
from typing import List, Any

In [4]:
import sys
print(sys.executable)

/workspaces/buidling_AI_agents/aihero/project/.venv/bin/python


In [5]:
from pydantic_ai import Agent
print("import works")

import works


In [6]:
from pathlib import Path
import json

project_root = Path("..")
with open(project_root / "my_chunks_sections.json", "r", encoding="utf-8") as f:
    my_chunks_sections = json.load(f)

len(my_chunks_sections)
my_chunks_sections[0]

{'filename': 'binance-spot-api-docs-master/CHANGELOG_CN.md',
 'section': '## REST API\n\n- `POST /api/v3/userDataStream`\n- `PUT /api/v3/userDataStream`\n- `DELETE /api/v3/userDataStream`'}

In [7]:
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np

print("all imports work")

all imports work


In [8]:
def normalize_binance_chunks(chunks):
    normalized = []

    for i, doc in enumerate(chunks):
        item = dict(doc)

        chunk_text = (
            item.get("chunk")
            or item.get("content")
            or item.get("text")
            or ""
        )

        title = item.get("title", "") or item.get("heading", "") or ""
        filename = (
            item.get("filename", "")
            or item.get("file_path", "")
            or item.get("path", "")
            or item.get("source", "")
            or ""
        )
        section = item.get("section", "") or item.get("header", "") or ""
        description = item.get("description", "") or ""

        normalized.append({
            "id": item.get("id", str(i)),
            "chunk": chunk_text,
            "title": title,
            "filename": filename,
            "section": section,
            "description": description,
            "url": item.get("url", ""),
            "raw": item
        })

    return normalized

In [9]:
binance_docs = []

for i, d in enumerate(my_chunks_sections):
    binance_docs.append({
        "id": str(i),
        "chunk": d.get("chunk", "") if isinstance(d, dict) else "",
        "title": d.get("title", "") if isinstance(d, dict) else "",
        "description": d.get("description", "") if isinstance(d, dict) else "",
        "filename": d.get("filename", "") if isinstance(d, dict) else "",
        "section": d.get("section", "") if isinstance(d, dict) else ""
    })

len(binance_docs)

367

In [10]:
binance_docs[0]

{'id': '0',
 'chunk': '',
 'title': '',
 'description': '',
 'filename': 'binance-spot-api-docs-master/CHANGELOG_CN.md',
 'section': '## REST API\n\n- `POST /api/v3/userDataStream`\n- `PUT /api/v3/userDataStream`\n- `DELETE /api/v3/userDataStream`'}

In [11]:
text_index = Index(
    text_fields=["chunk", "title", "description", "filename", "section"],
    keyword_fields=[]
)

text_index.fit(binance_docs)

In [12]:
query = "How do I sign a Binance Spot API request?"
text_results = text_index.search(query, num_results=5)
text_results

[{'id': '170',
  'chunk': '',
  'title': '',
  'description': '',
  'filename': 'binance-spot-api-docs-master/fix-api.md',
  'section': '## General API Information\n\n* FIX connections require TLS encryption. Please either use native TCP+TLS connection or set up a local proxy such as [stunnel](https://www.stunnel.org/) to handle TLS encryption.\n* APIs have a timeout of 10 seconds when processing a request. If a response from the Matching Engine takes longer than this, the API responds with "Timeout waiting for response from backend server. Send status unknown; execution status unknown." [(-1007 TIMEOUT)](errors.md#-1007-timeout)\n  * This does not always mean that the request failed in the Matching Engine.\n  * If the status of the request has not appeared in [User Data Stream](user-data-stream.md), please perform an API query for its status.\n* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.\

In [13]:
def preview_results(results, max_chars=300):
    for i, r in enumerate(results, 1):
        print(f"\nResult #{i}")
        print("title:      ", r.get("title", ""))
        print("section:    ", r.get("section", ""))
        print("filename:   ", r.get("filename", ""))
        print("chunk:      ", r.get("chunk", "")[:max_chars])
        print("-" * 80)

In [14]:
preview_results(text_results)


Result #1
title:       
section:     ## General API Information

* FIX connections require TLS encryption. Please either use native TCP+TLS connection or set up a local proxy such as [stunnel](https://www.stunnel.org/) to handle TLS encryption.
* APIs have a timeout of 10 seconds when processing a request. If a response from the Matching Engine takes longer than this, the API responds with "Timeout waiting for response from backend server. Send status unknown; execution status unknown." [(-1007 TIMEOUT)](errors.md#-1007-timeout)
  * This does not always mean that the request failed in the Matching Engine.
  * If the status of the request has not appeared in [User Data Stream](user-data-stream.md), please perform an API query for its status.
* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.
* To ensure uninterrupted connectivity, please make sure that your client sends **SNI (Server Name Indica

In [15]:
def normalize_binance_chunks(chunks):
    normalized = []

    for i, doc in enumerate(chunks):
        item = dict(doc)

        chunk_text = (
            item.get("chunk")
            or item.get("content")
            or item.get("text")
            or ""
        )

        title = item.get("title", "") or item.get("heading", "") or ""
        filename = (
            item.get("filename", "")
            or item.get("file_path", "")
            or item.get("path", "")
            or item.get("source", "")
            or ""
        )
        section = item.get("section", "") or item.get("header", "") or ""
        description = item.get("description", "") or ""

        normalized.append({
            "id": item.get("id", str(i)),
            "chunk": chunk_text,
            "title": title,
            "filename": filename,
            "section": section,
            "description": description,
            "url": item.get("url", ""),
            "raw": item
        })

    return normalized

normalized_chunks = normalize_binance_chunks(my_chunks_sections)

embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [16]:
binance_embeddings = []

for d in tqdm(binance_docs):
    v = embedding_model.encode(d["chunk"])
    binance_embeddings.append(v)

binance_embeddings = np.array(binance_embeddings)
binance_embeddings.shape

  0%|          | 0/367 [00:00<?, ?it/s]

(367, 768)

In [17]:
vector_index = VectorSearch()
vector_index.fit(binance_embeddings, binance_docs)

In [18]:
query = "How can I authenticate a private Binance API endpoint?"
q = embedding_model.encode(query)

vector_results = vector_index.search(q, num_results=5)
preview_results(vector_results)


Result #1
title:       
section:     ## 如何正确在本地维护一个order book副本

1. 打开与 `wss://stream.binance.com:9443/ws/bnbbtc@depth` 的 WebSocket 连接。
2. 开始缓存收到的event。请记录您收到的第一个event的 `U`值。
3. 访问 `https://api.binance.com/api/v3/depth?symbol=BNBBTC&limit=5000` 获取深度快照。
4. 如果快照中的 `lastUpdateId` 小于等于步骤 2 中的 `U` 值，请返回步骤 3。
5. 在收到的event中，丢弃快照中 `u` <= `lastUpdateId` 的所有event。现在第一个event的 `lastUpdateId` 应该在 `[U;u]` 范围以内。
6. 将本地order book设置为快照。它的更新ID 为 `lastUpdateId`。
7. 更新所有缓存的event，以及后续的所有event。

要将一个event应用于您的本地order book，请遵循以下更新过程：
1. 判断是否需要处理event：
    * 如果event的最后一次更新ID（`u`）小于本地order book的更新ID，忽略该event。
    * 如果event的首次更新ID（`U`）大于本地order book的更新ID加1，说明你错过了一些events。<br>请丢弃您的本地order book并从头开始重新同步。
    * 通常，下一event的`U`等于上一event的`u + 1`。
1. 对买价（`b`）和卖价（`a`）中的每个价位，设置order book中的新数量：
    * 如果该价位在order book中不存在，则插入该价位及其数量。
    * 如果数量为零，则从order book中删除此价位。
1. 将order book的更新ID设置为已处理event的最后一次更新ID（`u`）。

> [!NOTE]
> 由于从 API 检索的深度快照对价位的数量有限制（每侧最多 5000 个），因此除非它们发生变化，否则您将无法了解初始快照之外的价位数量。<br>
> 因此，在使用这些级别的信息时要小心，因为它们

In [19]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    q = embedding_model.encode(query)
    return vector_index.search(q, num_results=num_results)

In [20]:
def hybrid_search(query: str, num_results: int = 5, text_k: int = 10, vector_k: int = 10):
    text_results = text_search(query, num_results=text_k)
    vector_results = vector_search(query, num_results=vector_k)

    scores = {}

    for rank, r in enumerate(text_results):
        rid = r.get("id") or r.get("chunk_id") or str(r)
        if rid not in scores:
            scores[rid] = {"doc": r, "score": 0.0}
        scores[rid]["score"] += 1.0 / (rank + 1)

    for rank, r in enumerate(vector_results):
        rid = r.get("id") or r.get("chunk_id") or str(r)
        if rid not in scores:
            scores[rid] = {"doc": r, "score": 0.0}
        scores[rid]["score"] += 1.0 / (rank + 1)

    ranked = sorted(scores.values(), key=lambda x: x["score"], reverse=True)
    return [x["doc"] for x in ranked[:num_results]]

In [21]:
query = "What does recvWindow mean in Binance API requests?"
hybrid_results = hybrid_search(query, num_results=5)
preview_results(hybrid_results)


Result #1
title:       
section:     ## General API Information

* The following base endpoints are available. Please use whichever works best for your setup:
  * **https://api.binance.com**
  * **https://api-gcp.binance.com**
  * **https://api1.binance.com**
  * **https://api2.binance.com**
  * **https://api3.binance.com**
  * **https://api4.binance.com**
* The last 4 endpoints in the point above (`api1`-`api4`) should give better performance but have less stability.
* Responses are in JSON by default. To receive responses in SBE, refer to the [SBE FAQ](./faqs/sbe_faq.md) page.
* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.
* Some endpoints may return asset and/or symbol names containing non-ASCII characters encoded in UTF-8 even if the request did not contain non-ASCII characters.
* Data is returned in **chronological order**, unless noted otherwise.
  * Without `startTime` or `endTime`, 

In [22]:
test_queries = [
    "How do I sign a Binance Spot API request?",
    "What does recvWindow mean?",
    "Why am I getting timestamp error?",
    "How do I place a market order?",
    "How can I authenticate a private endpoint?",
    "What are the rate limits?",
    "How do I get exchange info?"
]

for query in test_queries:
    print("\n" + "=" * 100)
    print("QUERY:", query)
    results = hybrid_search(query, num_results=3)
    preview_results(results, max_chars=220)


QUERY: How do I sign a Binance Spot API request?

Result #1
title:       
section:     ## General API Information

* FIX connections require TLS encryption. Please either use native TCP+TLS connection or set up a local proxy such as [stunnel](https://www.stunnel.org/) to handle TLS encryption.
* APIs have a timeout of 10 seconds when processing a request. If a response from the Matching Engine takes longer than this, the API responds with "Timeout waiting for response from backend server. Send status unknown; execution status unknown." [(-1007 TIMEOUT)](errors.md#-1007-timeout)
  * This does not always mean that the request failed in the Matching Engine.
  * If the status of the request has not appeared in [User Data Stream](user-data-stream.md), please perform an API query for its status.
* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.
* To ensure uninterrupted connectivity, please make sur

In [23]:
len(my_chunks_sections)

367

In [24]:
preview_results(text_search("What does recvWindow mean?"))


Result #1
title:       
section:     ## In the API response, there's a field called `workingFloor`. What does that field mean?

This is a term used to determine where the order's last activity occurred (filling, expiring, or being placed as new, etc.).

If the `workingFloor` is `SOR`, this means your order interacted with other eligible order books in the SOR configuration.

If the `workingFloor` is `EXCHANGE`, this means your order interacted on the order book that you sent that order to.
filename:    binance-spot-api-docs-master/faqs/sor_faq.md
chunk:       
--------------------------------------------------------------------------------

Result #2
title:       
section:     ## What does the Price Range Execution Rule do?

This rule ensures that trades may only be executed at prices within and equal to a price range around a reference price.
filename:    binance-spot-api-docs-master/faqs/price_range_execution_rules.md
chunk:       
---------------------------------------------------

In [25]:
preview_results(vector_search("How do I authenticate a private endpoint?"))


Result #1
title:       
section:     ## 如何正确在本地维护一个order book副本

1. 打开与 `wss://stream.binance.com:9443/ws/bnbbtc@depth` 的 WebSocket 连接。
2. 开始缓存收到的event。请记录您收到的第一个event的 `U`值。
3. 访问 `https://api.binance.com/api/v3/depth?symbol=BNBBTC&limit=5000` 获取深度快照。
4. 如果快照中的 `lastUpdateId` 小于等于步骤 2 中的 `U` 值，请返回步骤 3。
5. 在收到的event中，丢弃快照中 `u` <= `lastUpdateId` 的所有event。现在第一个event的 `lastUpdateId` 应该在 `[U;u]` 范围以内。
6. 将本地order book设置为快照。它的更新ID 为 `lastUpdateId`。
7. 更新所有缓存的event，以及后续的所有event。

要将一个event应用于您的本地order book，请遵循以下更新过程：
1. 判断是否需要处理event：
    * 如果event的最后一次更新ID（`u`）小于本地order book的更新ID，忽略该event。
    * 如果event的首次更新ID（`U`）大于本地order book的更新ID加1，说明你错过了一些events。<br>请丢弃您的本地order book并从头开始重新同步。
    * 通常，下一event的`U`等于上一event的`u + 1`。
1. 对买价（`b`）和卖价（`a`）中的每个价位，设置order book中的新数量：
    * 如果该价位在order book中不存在，则插入该价位及其数量。
    * 如果数量为零，则从order book中删除此价位。
1. 将order book的更新ID设置为已处理event的最后一次更新ID（`u`）。

> [!NOTE]
> 由于从 API 检索的深度快照对价位的数量有限制（每侧最多 5000 个），因此除非它们发生变化，否则您将无法了解初始快照之外的价位数量。<br>
> 因此，在使用这些级别的信息时要小心，因为它们

In [26]:
preview_results(hybrid_search("Why am I getting timestamp error?"))


Result #1
title:       
section:     ## Error Codes

* Any endpoint can return an ERROR

Sample Payload below:
```javascript
{
    "code": -1121,
    "msg": "Invalid symbol."
}
```
* Specific error codes and messages are defined in [Errors Codes](./errors.md).
filename:    binance-spot-api-docs-master/rest-api.md
chunk:       
--------------------------------------------------------------------------------

Result #2
title:       
section:     ## REST API

- `POST /api/v3/userDataStream`
- `PUT /api/v3/userDataStream`
- `DELETE /api/v3/userDataStream`
filename:    binance-spot-api-docs-master/CHANGELOG_CN.md
chunk:       
--------------------------------------------------------------------------------

Result #3
title:       
section:     ## Error Codes

* Any endpoint can return an ERROR

Sample Payload below:
```javascript
{
    "code": -1121,
    "msg": "Invalid symbol."
}
```
* Specific error codes and messages are defined in [Errors Codes](./errors.md).
filename:    binance-spot-

In [27]:
#Text search works well for exact Binance terms like recvWindow, timestamp, and endpoint names.
#TVector search helps when the question is paraphrased.
#THybrid search is the safest overall.
#TFor this project, I would start with text search and keep hybrid as the stronger production option.

In [28]:
print([name for name in dir() if "search" in name.lower()])

['VectorSearch', 'hybrid_search', 'text_search', 'vector_search']


In [29]:
import inspect

print(inspect.signature(text_search))
print(inspect.signature(vector_search))
print(inspect.signature(hybrid_search))

(query, num_results=5)
(query, num_results=5)
(query: str, num_results: int = 5, text_k: int = 10, vector_k: int = 10)


In [30]:
def search_docs(query: str) -> str:
    """
    Search the Binance documentation using hybrid search and return
    structured context with metadata.
    """
    results = hybrid_search(query, num_results=5)

    formatted = []
    for i, r in enumerate(results, 1):
        if isinstance(r, dict):
            title = r.get("title", "")
            section = r.get("section", "")
            filename = r.get("filename", "")
            text = r.get("chunk") or r.get("text") or r.get("content") or ""
        else:
            title = ""
            section = ""
            filename = ""
            text = str(r)

        formatted.append(
            f"""Result {i}
title: {title}
section: {section}
filename: {filename}
content: {text}
"""
        )

    return "\n\n".join(formatted)

In [31]:
from pydantic_ai import Agent

system_prompt_v1 = """
You are a Binance documentation assistant.

Always use the search_docs tool before answering questions.
Base your answer only on the retrieved documentation.
If the answer is not found, say that clearly.
Do not guess.
"""

agent_v1 = Agent(
    name="binance_agent_v1",
    instructions=system_prompt_v1,
    tools=[search_docs],
    model="gpt-4o-mini"
)

/workspaces/buidling_AI_agents/aihero/project/.venv/lib/python3.12/site-packages/pydantic_ai/models/__init__.py:1280: DeprecationWarning: Specifying a model name without a provider prefix is deprecated. Instead of 'gpt-4o-mini', use 'openai:gpt-4o-mini'.
  provider_name, model_name = parse_model_id(model)


In [32]:
from pydantic_ai import Agent

system_prompt_v2 = """
You are a Binance documentation assistant.

Always use the search_docs tool before answering questions.
Base your answer on the retrieved Binance documentation.
Do not guess when the retrieved results are weak or unclear.

When answering:
- give a direct answer first
- mention important parameters or requirements
- mention authentication/signing requirements if relevant
- cite the filename or section used
""".strip()

agent_v2 = Agent(
    name="binance_agent_v2",
    instructions=system_prompt_v2,
    tools=[search_docs],
    model="gpt-4o-mini"
)


/workspaces/buidling_AI_agents/aihero/project/.venv/lib/python3.12/site-packages/pydantic_ai/models/__init__.py:1280: DeprecationWarning: Specifying a model name without a provider prefix is deprecated. Instead of 'gpt-4o-mini', use 'openai:gpt-4o-mini'.
  provider_name, model_name = parse_model_id(model)


In [33]:
from pydantic_ai import Agent

system_prompt_v3 = """
You are a Binance documentation assistant.

Rules:
1. Always call search_docs before answering.
2. Answer only from retrieved documentation.
3. If the retrieved context is weak, incomplete, or ambiguous, say so clearly.
4. Do not guess or invent Binance behavior.
5. Use the retrieved filename and section in your answer when available.

Only handle questions about:
- authentication
- signed requests
- timestamps
- recvWindow
- market orders
- limit orders
- required order parameters

If the question is outside these areas, say that it is outside the current assistant scope.

Answer format:
- Direct answer
- Key details / parameters
- Important caveats
- Source(s): filename / section
""".strip()

agent_v3 = Agent(
    name="binance_agent_v3",
    instructions=system_prompt_v3,
    tools=[search_docs],
    model="gpt-4o-mini"
)

In [34]:
import importlib
import logging_utils

importlib.reload(logging_utils)

from logging_utils import log_interaction_to_file

In [35]:
question = "How do I authenticate API requests?"

result_v1 = await agent_v1.run(question)
print("V1:\n", result_v1.output)

result_v2 = await agent_v2.run(question)
print("\nV2 ANSWER:")
print(result_v2.output)

log_file_v2 = log_interaction_to_file(
    agent_v2,
    result_v2.new_messages(),
    source="user",
    version="v2"
)
print("Saved V2 log:", log_file_v2)

result_v3 = await agent_v3.run(question)
print("\nV3 ANSWER:")
print(result_v3.output)

log_file_v3 = log_interaction_to_file(
    agent_v3,
    result_v3.new_messages(),
    source="user",
    version="v3"
)
print("Saved V3 log:", log_file_v3)

V1:
 To authenticate API requests on Binance, you can use session authentication for a WebSocket connection. Here are the steps:

1. **Log in with your API Key**:
   You need to send a `session.logon` request with your API key, signature, and timestamp. Here's an example request format:

   ```json
   {
       "id": "<unique_id>",
       "method": "session.logon",
       "params": {
           "apiKey": "<your_api_key>",
           "signature": "<your_signature>",
           "timestamp": <current_timestamp>
       }
   }
   ```

   Once authenticated, you won't need to specify the `apiKey` and `signature` in future requests for that session, but you still need to include a `timestamp` parameter for signed requests.

2. **Query session status**:
   You can check the status of your connection and the current API key in use by sending a `session.status` request:

   ```json
   {
       "id": "<unique_id>",
       "method": "session.status"
   }
   ```

3. **Log out of the session**:
   If

In [36]:
from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from logging_utils import log_interaction_to_file

In [37]:
import importlib
import logging_utils

importlib.reload(logging_utils)

from logging_utils import log_interaction_to_file

print("logging_utils reloaded")

logging_utils reloaded


In [38]:
eval_questions = [
    "How do I authenticate API requests?",
    "Which Binance API endpoints require signed requests?",
    "What does recvWindow mean in Binance API requests?",
    "Why am I getting a timestamp error on Binance?",
    "How do I place a market order?",
    "What parameters are required to create an order?",
    "What is the difference between a market order and a limit order?",
    "How do I sign a Binance Spot API request?",
    "What fields are needed for a LIMIT order?",
    "What fields are needed for a MARKET order?"
]

In [39]:
import logging_utils
print(logging_utils.__file__)

/workspaces/buidling_AI_agents/aihero/project/logging_utils.py


In [40]:
import inspect
from logging_utils import log_interaction_to_file

print(inspect.signature(log_interaction_to_file))

(agent, messages, source='user', version=None)


In [41]:
import importlib
import logging_utils

importlib.reload(logging_utils)
from logging_utils import log_interaction_to_file

In [42]:
import inspect
print(inspect.signature(log_interaction_to_file))

(agent, messages, source='user', version=None)


In [43]:
from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import logging_utils
print("Imported from:", logging_utils.__file__)

Imported from: /workspaces/buidling_AI_agents/aihero/project/logging_utils.py


In [44]:
import importlib
import inspect

importlib.reload(logging_utils)

print("log_entry:", inspect.signature(logging_utils.log_entry))
print("log_interaction_to_file:", inspect.signature(logging_utils.log_interaction_to_file))

log_entry: (agent, messages, source='user', version=None)
log_interaction_to_file: (agent, messages, source='user', version=None)


In [45]:
from pathlib import Path

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

for f in LOG_DIR.glob("*.json"):
    f.unlink()

print("logs cleared")

logs cleared


In [46]:
for q in eval_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", q)

    result_v3 = await agent_v3.run(q)
    print("\nV3 ANSWER:")
    print(result_v3.output)

    log_file_v3 = log_interaction_to_file(
        agent_v3,
        result_v3.new_messages(),
        source="user",
        version="v3"
    )
    print("Saved V3 log:", log_file_v3)


QUESTION: How do I authenticate API requests?



V3 ANSWER:
To authenticate API requests on Binance, you need to follow certain steps that include using your API key, generating a signature, and including a timestamp in your requests.

### Direct Answer
Authenticate your API requests by providing the following parameters in a signed request:
1. **apiKey**: Your unique API key.
2. **signature**: A HMAC SHA256 signature generated using your secret key and other parameters.
3. **timestamp**: A timestamp indicating the request time to prevent replay attacks. This must be included for signed requests.

### Key Details / Parameters
- **session.logon**: A method to authenticate WebSocket connection using your API key.
- Required parameters for `session.logon`:
  - `apiKey` (mandatory)
  - `signature` (mandatory)
  - `timestamp` (mandatory)
  - `recvWindow` (optional, must not exceed 60000 ms)

### Important Caveats
- Only **Ed25519** keys are supported.
- After authenticating with `session.logon`, you can omit `apiKey` and `signature` in s

In [47]:

import json
from pydantic import BaseModel
from pydantic_ai import Agent


class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool


class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>)
to a user question (<QUESTION>).
We also include the full interaction log (<LOG>) for analysis.

For each item, check if the condition is met.

Checklist:
- instructions_follow: The agent followed the system instructions
- answer_relevant: The response directly answers the user's question
- answer_clear: The answer is easy to understand and technically correct
- answer_citations: The answer includes references or source filenames/sections when appropriate
- completeness: The answer covers the key parts of the user's request
- tool_call_search: The search_docs tool was used before answering

Output true/false for each check and provide a short explanation.
""".strip()


eval_agent = Agent(
    name="binance_eval_agent",
    model="gpt-4o-mini",
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist,
)


def load_log_file(log_file):
    with open(log_file, "r", encoding="utf-8") as f_in:
        log_data = json.load(f_in)
    log_data["log_file"] = str(log_file)
    return log_data


def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []

        for original_part in m["parts"]:
            part = original_part.copy()
            kind = part["part_kind"]

            if kind == "user-prompt" and "timestamp" in part:
                del part["timestamp"]

            if kind == "tool-call" and "tool_call_id" in part:
                del part["tool_call_id"]

            if kind == "tool-return":
                if "tool_call_id" in part:
                    del part["tool_call_id"]
                if "metadata" in part:
                    del part["metadata"]
                if "timestamp" in part:
                    del part["timestamp"]
                part["content"] = "RETURN_RESULTS_REDACTED"

            if kind == "text" and "id" in part:
                del part["id"]

            parts.append(part)

        log_simplified.append({
            "kind": m["kind"],
            "parts": parts
        })

    return log_simplified


user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()


async def evaluate_log_record(log_record):
    messages = log_record["messages"]
    instructions = log_record["system_prompt"]
    question = messages[0]["parts"][0]["content"]
    answer = messages[-1]["parts"][0]["content"]

    log_simplified = simplify_log_messages(messages)
    log_json = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log_json,
    )

    result = await eval_agent.run(user_prompt)
    return result.output

/workspaces/buidling_AI_agents/aihero/project/.venv/lib/python3.12/site-packages/pydantic_ai/models/__init__.py:1280: DeprecationWarning: Specifying a model name without a provider prefix is deprecated. Instead of 'gpt-4o-mini', use 'openai:gpt-4o-mini'.
  provider_name, model_name = parse_model_id(model)


In [48]:
from pathlib import Path

LOG_DIR = Path("logs")
print("Using LOG_DIR:", LOG_DIR.resolve())

log_files = list(LOG_DIR.glob("*.json"))[:5]
print("Found log files:", len(log_files))
print(log_files[:5])

Using LOG_DIR: /workspaces/buidling_AI_agents/aihero/project/notebooks/logs
Found log files: 5
[PosixPath('logs/binance_agent_v3_v3_20260331_140442_a07462.json'), PosixPath('logs/binance_agent_v3_v3_20260331_140504_6e725a.json'), PosixPath('logs/binance_agent_v3_v3_20260331_140443_67af0f.json'), PosixPath('logs/binance_agent_v3_v3_20260331_140456_d17d45.json'), PosixPath('logs/binance_agent_v3_v3_20260331_140406_b8278c.json')]


In [49]:
from pathlib import Path

LOG_DIR = (Path("..") / "logs").resolve()
print("Using LOG_DIR:", LOG_DIR)

Using LOG_DIR: /workspaces/buidling_AI_agents/aihero/project/logs


In [50]:
log_files = list(LOG_DIR.glob("*.json"))
print("Found:", len(log_files))
print(log_files[:3])

Found: 20
[PosixPath('/workspaces/buidling_AI_agents/aihero/project/logs/binance_agent_v2_20260331_124955_f8a2d5.json'), PosixPath('/workspaces/buidling_AI_agents/aihero/project/logs/binance_agent_v2_20260331_125038_eda0ce.json'), PosixPath('/workspaces/buidling_AI_agents/aihero/project/logs/binance_agent_v2_20260331_124958_8a22a4.json')]


In [51]:
log_files = list(LOG_DIR.glob("*.json"))[:5]

eval_results = []

for log_file in log_files:
    log_record = load_log_file(log_file)
    eval_result = await evaluate_log_record(log_record)
    eval_results.append((log_record, eval_result))

print("Done:", len(eval_results))

Done: 5


In [52]:
log_record = load_log_file(log_files[0])
eval_result = await evaluate_log_record(log_record)

print(eval_result.summary)
for check in eval_result.checklist:
    print(check.check_name, "=>", check.check_pass)
    print("reason:", check.justification)
    print()

The agent followed the instructions and was relevant but did not provide a clear, comprehensive answer, nor did it cite any references.
instructions_follow => True
reason: The agent should have used the search_docs tool before answering, which it did.

answer_relevant => True
reason: The answer addresses the user's question about the `balanceUpdate` event.

answer_clear => False
reason: The answer is mostly clear, explaining what the event typically does, but does not provide specific details as intended.

answer_citations => False
reason: The answer fails to cite specific sections or filenames from the Binance documentation.

completeness => False
reason: While the answer provides a general overview, it lacks crucial details about the fields and data structure related to `balanceUpdate`, making it incomplete.

tool_call_search => True
reason: The search_docs tool was indeed used to retrieve information before answering the question.



In [53]:
eval_results = []
failed_logs = []

for log_file in list(LOG_DIR.glob("*.json")):
    try:
        log_record = load_log_file(log_file)
        eval_result = await evaluate_log_record(log_record)
        eval_results.append((log_record, eval_result))
        print("Evaluated:", log_file.name)
    except Exception as e:
        failed_logs.append((log_file.name, str(e)))
        print("Failed:", log_file.name, e)

print("Success:", len(eval_results))
print("Failed:", len(failed_logs))

Evaluated: binance_agent_v2_20260331_124955_f8a2d5.json
Evaluated: binance_agent_v2_20260331_125038_eda0ce.json
Evaluated: binance_agent_v2_20260331_124958_8a22a4.json
Evaluated: binance_agent_v2_20260331_125004_8ea489.json
Evaluated: binance_agent_v2_20260331_124938_9736d1.json
Evaluated: binance_agent_v2_20260331_125044_ae7442.json
Evaluated: binance_agent_v2_20260331_125011_8fd8cf.json
Evaluated: binance_agent_v2_20260331_124942_288266.json
Evaluated: binance_agent_v2_20260331_125055_f77a09.json
Evaluated: binance_agent_v2_20260331_124927_1536ee.json
Evaluated: binance_agent_v2_20260331_125019_3f44e6.json
Evaluated: binance_agent_v2_20260331_125030_ed468c.json
Evaluated: binance_agent_v2_20260331_124951_2661d8.json
Evaluated: binance_agent_v2_20260331_125106_428d9e.json
Evaluated: binance_agent_v2_20260331_124935_9ad4d5.json
Evaluated: binance_agent_v2_20260331_125050_f8564d.json
Evaluated: binance_agent_v2_20260331_125102_133a0e.json
Evaluated: binance_agent_v2_20260331_124946_da5a

In [54]:
from pathlib import Path
import pandas as pd

rows = []

for log_record, eval_result in eval_results:
    messages = log_record["messages"]

    row = {
        "file": Path(log_record["log_file"]).name,
        "agent_name": log_record.get("agent_name"),
        "agent_version": log_record.get("agent_version"),
        "source": log_record.get("source"),
        "question": messages[0]["parts"][0]["content"],
        "answer": messages[-1]["parts"][0]["content"],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("evaluation_results.csv", index=False)

print("saved evaluation_results.csv")
print(df["source"].value_counts())

saved evaluation_results.csv
source
ai-generated    20
Name: count, dtype: int64


In [109]:
score_cols = [
    "instructions_follow",
    "answer_relevant",
    "answer_clear",
    "answer_citations",
    "completeness",
    "tool_call_search"
]

In [110]:
df_user = df[df["source"] == "user"]
df_ai = df[df["source"] == "ai-generated"]

print("User questions:")
print(df_user.mean(numeric_only=True))

print("\nAI-generated questions:")
print(df_ai.mean(numeric_only=True))

User questions:
instructions_follow    1.0
answer_relevant        1.0
answer_clear           0.9
answer_citations       0.9
completeness           0.9
tool_call_search       1.0
dtype: float64

AI-generated questions:
instructions_follow    1.000000
answer_relevant        1.000000
answer_clear           1.000000
answer_citations       0.857143
completeness           0.857143
tool_call_search       1.000000
dtype: float64


In [111]:
%%writefile logging_utils.py
import json
import secrets
from pathlib import Path
from datetime import datetime
from pydantic_ai.messages import ModelMessagesTypeAdapter

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

def log_entry(agent, messages, source="user", version=None):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    provider = None
    model_name = None
    try:
        provider = agent.model.system
    except Exception:
        pass

    try:
        model_name = agent.model.model_name
    except Exception:
        model_name = str(agent.model)

    return {
        "agent_name": agent.name,
        "agent_version": version,
        "system_prompt": getattr(agent, "_instructions", None),
        "provider": provider,
        "model": model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source,
    }

def log_interaction_to_file(agent, messages, source="user", version=None):
    entry = log_entry(agent, messages, source=source, version=version)

    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{version or 'na'}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath

Overwriting logging_utils.py


In [112]:
import importlib
import logging_utils

importlib.reload(logging_utils)

from logging_utils import log_interaction_to_file

In [113]:
from importlib import reload
import logging_utils

reload(logging_utils)
from logging_utils import log_interaction_to_file

print("logging utils reloaded")

logging utils reloaded


In [114]:
from pathlib import Path
list(Path("logs").glob("*.json"))

[PosixPath('logs/binance_agent_v3_v3_20260331_140442_a07462.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_141438_f3f81c.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_141459_0fab35.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_141406_b0553d.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_140504_6e725a.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_141426_b7daa5.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_140443_67af0f.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_140456_d17d45.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_140406_b8278c.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_140424_63fcb9.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_140434_d98ca6.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_141429_6ac602.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_140428_43035e.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_140449_74f0be.json'),
 PosixPath('logs/binance_agent_v3_v3_20260331_14

In [115]:
from evals import load_log_file, evaluate_log_record
from pathlib import Path
import pandas as pd

LOG_DIR = Path("logs")

eval_results = []
failed_logs = []

for log_file in list(LOG_DIR.glob("*.json")):
    try:
        log_record = load_log_file(log_file)
        eval_result = await evaluate_log_record(log_record)
        eval_results.append((log_record, eval_result))
        print("Evaluated:", log_file.name)
    except Exception as e:
        failed_logs.append((log_file.name, str(e)))
        print("Failed:", log_file.name, e)

print("Success:", len(eval_results))
print("Failed:", len(failed_logs))

rows = []

for log_record, eval_result in eval_results:
    messages = log_record["messages"]

    row = {
        "agent_name": log_record.get("agent_name"),
        "agent_version": log_record.get("agent_version"),
        "source": log_record.get("source"),
        "question": messages[0]["parts"][0]["content"],
        "answer": messages[-1]["parts"][0]["content"],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("evaluation_results.csv", index=False)

print(df["source"].value_counts())
print(df["agent_version"].value_counts())

df.head()

Evaluated: binance_agent_v3_v3_20260331_140442_a07462.json
Evaluated: binance_agent_v3_v3_20260331_141438_f3f81c.json
Evaluated: binance_agent_v3_v3_20260331_141459_0fab35.json
Evaluated: binance_agent_v3_v3_20260331_141406_b0553d.json
Evaluated: binance_agent_v3_v3_20260331_140504_6e725a.json
Evaluated: binance_agent_v3_v3_20260331_141426_b7daa5.json
Evaluated: binance_agent_v3_v3_20260331_140443_67af0f.json
Evaluated: binance_agent_v3_v3_20260331_140456_d17d45.json
Evaluated: binance_agent_v3_v3_20260331_140406_b8278c.json
Evaluated: binance_agent_v3_v3_20260331_140424_63fcb9.json
Evaluated: binance_agent_v3_v3_20260331_140434_d98ca6.json
Evaluated: binance_agent_v3_v3_20260331_141429_6ac602.json
Evaluated: binance_agent_v3_v3_20260331_140428_43035e.json
Evaluated: binance_agent_v3_v3_20260331_140449_74f0be.json
Evaluated: binance_agent_v3_v3_20260331_140402_eb272c.json
Evaluated: binance_agent_v3_v3_20260331_141347_8ea8c8.json
Evaluated: binance_agent_v3_v3_20260331_141354_3ebffd.js

,agent_name,agent_version,source,question,answer,instructions_follow,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
0,binance_agent_v3,v3,user,What parameters are required to create an order?,"To create an order on Binance, the required pa...",True,True,True,True,True,True
1,binance_agent_v3,v3,ai-generated,What are the required parameters when placing ...,The required parameters for placing a limit or...,True,True,True,True,True,True
2,binance_agent_v3,v3,ai-generated,Are there any specific field requirements for ...,Market orders in the Binance API have specific...,True,True,True,True,True,True
3,binance_agent_v3,v3,ai-generated,How do I ensure that the timestamp I send in m...,To ensure that the timestamp you send in your ...,True,True,True,True,True,True
4,binance_agent_v3,v3,user,What fields are needed for a MARKET order?,"For a MARKET order on Binance, the required fi...",True,True,True,True,True,True


In [116]:
log_files = list(LOG_DIR.glob("*.json"))
print("Found:", len(log_files))
print(log_files[:5])

log_record = load_log_file(log_files[0])
eval_result = await evaluate_log_record(log_record)

print(eval_result.summary)
for check in eval_result.checklist:
    print(check.check_name, "=>", check.check_pass)
    print("reason:", check.justification)
    print()

Found: 17
[PosixPath('logs/binance_agent_v3_v3_20260331_140442_a07462.json'), PosixPath('logs/binance_agent_v3_v3_20260331_141438_f3f81c.json'), PosixPath('logs/binance_agent_v3_v3_20260331_141459_0fab35.json'), PosixPath('logs/binance_agent_v3_v3_20260331_141406_b0553d.json'), PosixPath('logs/binance_agent_v3_v3_20260331_140504_6e725a.json')]


The AI agent's answer satisfies all the checklist conditions.
instructions_follow => True
reason: The agent called search_docs before answering as required by the system instructions.

answer_relevant => True
reason: The response directly addresses the user's question about the required parameters for creating an order.

answer_clear => True
reason: The answer is clearly structured and provides an easy-to-understand explanation of the parameters.

answer_citations => True
reason: The answer includes a citation from the Binance documentation indicating the source of the information provided.

completeness => True
reason: The answer covers the key required parameters and also mentions optional parameters and important caveats.

tool_call_search => True
reason: The search_docs tool was used multiple times before the final answer was provided.



In [117]:
import pandas as pd
from pathlib import Path

rows = []

for log_record, eval_result in eval_results:
    messages = log_record["messages"]

    row = {
        "file": Path(log_record["log_file"]).name,
        "agent_name": log_record.get("agent_name"),
        "agent_version": log_record.get("agent_version"),
        "source": log_record.get("source"),
        "question": messages[0]["parts"][0]["content"],
        "answer": messages[-1]["parts"][0]["content"],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)
    rows.append(row)

df_evals = pd.DataFrame(rows)
df_evals.to_csv("evaluation_results.csv", index=False)
print("saved evaluation_results.csv")

saved evaluation_results.csv


In [118]:
df = pd.DataFrame(rows)
df.to_csv("evaluation_results.csv", index=False)

In [119]:
df.mean(numeric_only=True)

instructions_follow    1.000000
answer_relevant        0.941176
answer_clear           1.000000
answer_citations       0.882353
completeness           0.882353
tool_call_search       1.000000
dtype: float64

In [120]:
from importlib import reload
import logging_utils
import evals

reload(logging_utils)
reload(evals)

from logging_utils import log_interaction_to_file
from evals import load_log_file, evaluate_log_record

print("modules reloaded")

modules reloaded


/workspaces/buidling_AI_agents/aihero/project/.venv/lib/python3.12/site-packages/pydantic_ai/models/__init__.py:1280: DeprecationWarning: Specifying a model name without a provider prefix is deprecated. Instead of 'gpt-4o-mini', use 'openai:gpt-4o-mini'.
  provider_name, model_name = parse_model_id(model)


In [121]:
user_df = df[df["source"] == "user"].copy()
ai_df = df[df["source"] == "ai-generated"].copy()

In [122]:
print("Source distribution:")
print(df["source"].value_counts(dropna=False))

print("\nUser rows:", len(user_df))
print("AI rows:", len(ai_df))

print("\nUser sample:")
print(user_df[score_cols].head())

print("\nFailed logs sample:")
print(failed_logs[:5])

Source distribution:
source
user            10
ai-generated     7
Name: count, dtype: int64

User rows: 10
AI rows: 7

User sample:
   instructions_follow  answer_relevant  answer_clear  answer_citations  \
0                 True             True          True              True   
4                 True             True          True              True   
6                 True             True          True              True   
7                 True             True          True              True   
8                 True             True          True              True   

   completeness  tool_call_search  
0          True              True  
4          True              True  
6          True              True  
7          True              True  
8          True              True  

Failed logs sample:
[]


In [123]:
df_v1 = df_evals[df_evals["agent_version"] == "v1"]
df_v2 = df_evals[df_evals["agent_version"] == "v2"]

print("V1 rows:", len(df_v1))
print("V2 rows:", len(df_v2))

print("\nV1 averages:")
print(df_v1.mean(numeric_only=True))

print("\nV2 averages:")
print(df_v2.mean(numeric_only=True))

V1 rows: 0
V2 rows: 0

V1 averages:
instructions_follow   NaN
answer_relevant       NaN
answer_clear          NaN
answer_citations      NaN
completeness          NaN
tool_call_search      NaN
dtype: float64

V2 averages:
instructions_follow   NaN
answer_relevant       NaN
answer_clear          NaN
answer_citations      NaN
completeness          NaN
tool_call_search      NaN
dtype: float64


In [124]:
failure_cols = [
    "instructions_follow",
    "answer_relevant",
    "answer_clear",
    "answer_citations",
    "completeness",
    "tool_call_search"
]

failed_rows = df_evals[
    ~(df_evals[failure_cols].all(axis=1))
]

failed_rows[["file", "question"] + failure_cols]

,file,question,instructions_follow,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
10,binance_agent_v3_v3_20260331_140434_d98ca6.json,How do I place a market order?,True,True,True,False,False,True
11,binance_agent_v3_v3_20260331_141429_6ac602.json,Can you explain the difference between market ...,True,False,True,False,False,True


In [125]:
df_evals[df_evals["answer_citations"] == False][["file", "question", "answer"]]

,file,question,answer
10,binance_agent_v3_v3_20260331_140434_d98ca6.json,How do I place a market order?,The retrieved documentation does not provide s...
11,binance_agent_v3_v3_20260331_141429_6ac602.json,Can you explain the difference between market ...,The retrieved documentation does not contain s...


In [126]:
df_evals[df_evals["completeness"] == False][["file", "question", "answer"]]

,file,question,answer
10,binance_agent_v3_v3_20260331_140434_d98ca6.json,How do I place a market order?,The retrieved documentation does not provide s...
11,binance_agent_v3_v3_20260331_141429_6ac602.json,Can you explain the difference between market ...,The retrieved documentation does not contain s...


In [127]:
df_v1.to_csv("evaluation_results_v1.csv", index=False)
df_v2.to_csv("evaluation_results_v2.csv", index=False)

In [128]:
print("V1")
print(df_v1.mean(numeric_only=True))

print("V2")
print(df_v2.mean(numeric_only=True))

V1
instructions_follow   NaN
answer_relevant       NaN
answer_clear          NaN
answer_citations      NaN
completeness          NaN
tool_call_search      NaN
dtype: float64
V2
instructions_follow   NaN
answer_relevant       NaN
answer_clear          NaN
answer_citations      NaN
completeness          NaN
tool_call_search      NaN
dtype: float64


In [129]:
print("\nV1 length:", len(result_v1.output))
print("V2 length:", len(result_v2.output))


V1 length: 1454
V2 length: 2028


In [130]:
from pathlib import Path

Path("evaluation_results.csv").exists()

True

In [131]:
import pandas as pd

df = pd.read_csv("evaluation_results.csv")
df.head()

,file,agent_name,agent_version,source,question,answer,instructions_follow,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
0,binance_agent_v3_v3_20260331_140442_a07462.json,binance_agent_v3,v3,user,What parameters are required to create an order?,"To create an order on Binance, the required pa...",True,True,True,True,True,True
1,binance_agent_v3_v3_20260331_141438_f3f81c.json,binance_agent_v3,v3,ai-generated,What are the required parameters when placing ...,The required parameters for placing a limit or...,True,True,True,True,True,True
2,binance_agent_v3_v3_20260331_141459_0fab35.json,binance_agent_v3,v3,ai-generated,Are there any specific field requirements for ...,Market orders in the Binance API have specific...,True,True,True,True,True,True
3,binance_agent_v3_v3_20260331_141406_b0553d.json,binance_agent_v3,v3,ai-generated,How do I ensure that the timestamp I send in m...,To ensure that the timestamp you send in your ...,True,True,True,True,True,True
4,binance_agent_v3_v3_20260331_140504_6e725a.json,binance_agent_v3,v3,user,What fields are needed for a MARKET order?,"For a MARKET order on Binance, the required fi...",True,True,True,True,True,True


In [132]:
df.shape

(17, 12)

In [133]:
df_v1 = df[df["agent_version"] == "v1"]
df_v2 = df[df["agent_version"] == "v2"]

print("V1:")
print(df_v1.mean(numeric_only=True))

print("\nV2:")
print(df_v2.mean(numeric_only=True))

V1:
instructions_follow   NaN
answer_relevant       NaN
answer_clear          NaN
answer_citations      NaN
completeness          NaN
tool_call_search      NaN
dtype: float64

V2:
instructions_follow   NaN
answer_relevant       NaN
answer_clear          NaN
answer_citations      NaN
completeness          NaN
tool_call_search      NaN
dtype: float64


In [134]:
df[df["completeness"] == False]

,file,agent_name,agent_version,source,question,answer,instructions_follow,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
10,binance_agent_v3_v3_20260331_140434_d98ca6.json,binance_agent_v3,v3,user,How do I place a market order?,The retrieved documentation does not provide s...,True,True,True,False,False,True
11,binance_agent_v3_v3_20260331_141429_6ac602.json,binance_agent_v3,v3,ai-generated,Can you explain the difference between market ...,The retrieved documentation does not contain s...,True,False,True,False,False,True


In [135]:
df_user = df[df["source"] == "user"]
df_ai = df[df["source"] == "ai-generated"]

print("User questions:")
print(df_user.mean(numeric_only=True))

print("\nAI-generated questions:")
print(df_ai.mean(numeric_only=True))

User questions:
instructions_follow    1.0
answer_relevant        1.0
answer_clear           1.0
answer_citations       0.9
completeness           0.9
tool_call_search       1.0
dtype: float64

AI-generated questions:
instructions_follow    1.000000
answer_relevant        0.857143
answer_clear           1.000000
answer_citations       0.857143
completeness           0.857143
tool_call_search       1.000000
dtype: float64


In [136]:
df.mean(numeric_only=True)
#spot fake scores

instructions_follow    1.000000
answer_relevant        0.941176
answer_clear           1.000000
answer_citations       0.882353
completeness           0.882353
tool_call_search       1.000000
dtype: float64

In [137]:
import json
from pathlib import Path

LOG_DIR = Path("logs")
print("Using LOG_DIR:", LOG_DIR.resolve())

log_files = list(LOG_DIR.glob("*.json"))

print("Using:", LOG_DIR)
print("Found:", len(log_files))
print(log_files[:3])

if not log_files:
    raise ValueError("No log files found — wrong path")

with open(log_files[0], "r") as f:
    log = json.load(f)

log

Using LOG_DIR: /workspaces/buidling_AI_agents/aihero/project/notebooks/logs
Using: logs
Found: 17
[PosixPath('logs/binance_agent_v3_v3_20260331_140442_a07462.json'), PosixPath('logs/binance_agent_v3_v3_20260331_141438_f3f81c.json'), PosixPath('logs/binance_agent_v3_v3_20260331_141459_0fab35.json')]


{'agent_name': 'binance_agent_v3',
 'agent_version': 'v3',
 'system_prompt': ['You are a Binance documentation assistant.\n\nRules:\n1. Always call search_docs before answering.\n2. Answer only from retrieved documentation.\n3. If the retrieved context is weak, incomplete, or ambiguous, say so clearly.\n4. Do not guess or invent Binance behavior.\n5. Use the retrieved filename and section in your answer when available.\n\nOnly handle questions about:\n- authentication\n- signed requests\n- timestamps\n- recvWindow\n- market orders\n- limit orders\n- required order parameters\n\nIf the question is outside these areas, say that it is outside the current assistant scope.\n\nAnswer format:\n- Direct answer\n- Key details / parameters\n- Important caveats\n- Source(s): filename / section'],
 'provider': 'openai',
 'model': 'gpt-4o-mini',
 'tools': ['search_docs'],
 'messages': [{'parts': [{'content': 'What parameters are required to create an order?',
     'timestamp': '2026-03-31T14:04:34.

In [138]:
df["answer_length"] = df["answer"].apply(len)

df_v1 = df[df["agent_version"] == "v1"]
df_v2 = df[df["agent_version"] == "v2"]

print(df_v1["answer_length"].mean())
print(df_v2["answer_length"].mean())

nan
nan


In [139]:
df["source"].value_counts(dropna=False)

source
user            10
ai-generated     7
Name: count, dtype: int64

In [140]:
len(df[df["source"] == "ai-generated"])

7

In [141]:
import json
import random
from pydantic import BaseModel
from pydantic_ai import Agent

In [142]:
class QuestionsList(BaseModel):
    questions: list[str]

In [143]:
question_generation_prompt = """
Generate realistic evaluation questions for a Binance API documentation assistant.

Only generate questions about:
- authentication
- signed requests
- timestamps
- recvWindow
- market orders
- limit orders
- required order parameters

Avoid unrelated areas such as balances, trades, rate limits, exchange info, or account history.

Return one concise, realistic user question per input snippet.
""".strip()

In [144]:
question_generator = Agent(
    name="binance_question_generator",
    instructions=question_generation_prompt,
    model="gpt-4o-mini",
    output_type=QuestionsList,
)

/workspaces/buidling_AI_agents/aihero/project/.venv/lib/python3.12/site-packages/pydantic_ai/models/__init__.py:1280: DeprecationWarning: Specifying a model name without a provider prefix is deprecated. Instead of 'gpt-4o-mini', use 'openai:gpt-4o-mini'.
  provider_name, model_name = parse_model_id(model)


In [145]:
from pathlib import Path
import json
import random

project_root = Path("..")

with open(project_root / "my_chunks_sections.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

sample = random.sample(chunks, 30)
prompt_docs = [clean_section(d.get("section", "")) for d in sample if d.get("section")]

In [146]:
def clean_section(text: str) -> str:
    if not text:
        return ""
    return text.replace("\n", " ").strip()

sample = random.sample(chunks, 10)

prompt_docs = [
    clean_section(
        d.get("section") or d.get("chunk") or d.get("text") or ""
    )
    for d in sample
]

In [147]:
sample[0]

{'filename': 'binance-spot-api-docs-master/web-socket-api.md',
 'section': '## User Data Stream requests\n\n### User Data Stream subscription\n\n<a id=general_info_user_data_stream_subscriptions></a>\n**General information:**\n\n* [User Data Stream](user-data-stream.md) subscriptions allow you to receive all the events related to a given account on a WebSocket connection.\n* There are 2 ways to start a subscription:\n  * If you have an authenticated session, then you can subscribe to events for that authenticated account using [`userDataStream.subscribe`](#user-data-stream-subscribe).\n  * In any session, authenticated or not, you can subscribe to events for one or more accounts for which you can provide an API Key signature, using [`userDataStream.subscribe.signature`](#user-data-signature).\n  * You can have only one active subscription for a given account on a given connection.\n* Subscriptions are identified by a `subscriptionId` which is returned when starting the subscription. Th

In [148]:
def clean_section(text):
    return text.replace("\n", " ").strip()

prompt_docs = [clean_section(d["section"]) for d in sample]

In [149]:
prompt = json.dumps(prompt_docs, ensure_ascii=False)
result = await question_generator.run(prompt)
ai_questions = result.output.questions
ai_questions

['What is the process for creating a signed request with the Binance API?']

In [150]:
for q in ai_questions:
    print(q)

What is the process for creating a signed request with the Binance API?


In [151]:
for q in ai_questions:
    print("\nAI QUESTION:", q)

    result_v3 = await agent_v3.run(q)
    print("\nV3 ANSWER:")
    print(result_v3.output)

    log_file_v3 = log_interaction_to_file(
        agent_v3,
        result_v3.new_messages(),
        source="ai-generated",
        version="v3"
    )

    print("Saved V3:", log_file_v3)


AI QUESTION: What is the process for creating a signed request with the Binance API?

V3 ANSWER:
To create a signed request with the Binance API, follow these steps:

1. **Prepare the Request:**
   - Format the request as a JSON object containing the following fields:
     - `id`: An arbitrary ID to match responses to requests (can be UUID, timestamp, etc.).
     - `method`: The method name for the API call (e.g., `"order.place"`).
     - `params`: The parameters required for the request, should include necessary fields like `symbol`, `side`, `type`, `price`, `quantity`, `timeInForce`, `timestamp`, and `apiKey`.

2. **Add a Timestamp:**
   - Include a `timestamp` parameter that represents the time of the request. This must be in milliseconds.

3. **Generate the Signature:**
   - The `signature` is created by signing the query string of the parameters with your API secret. This signature is then added to the request.

4. **Example of a Signed Request:**
   Here’s how a signed request l

In [152]:
import json
from pathlib import Path
from collections import Counter

sources = []
for f in Path("logs").glob("*.json"):
    with open(f, "r", encoding="utf-8") as infile:
        sources.append(json.load(infile).get("source"))

Counter(sources)

Counter({'user': 10, 'ai-generated': 8})

In [153]:
df["source"].value_counts()

source
user            10
ai-generated     7
Name: count, dtype: int64

In [154]:
df_ai = df[df["source"] == "ai-generated"]
df_ai.mean(numeric_only=True)

instructions_follow       1.000000
answer_relevant           0.857143
answer_clear              1.000000
answer_citations          0.857143
completeness              0.857143
tool_call_search          1.000000
answer_length          1433.571429
dtype: float64

In [155]:
# user questions
df_user = df[df["source"] == "user"]

# AI-generated questions
df_ai = df[df["source"] == "ai-generated"]

# split by version
df_v1 = df[df["agent_version"] == "v1"]
df_v2 = df[df["agent_version"] == "v2"]

In [156]:
print("V1 performance:")
print(df_v1.mean(numeric_only=True))

print("\nV2 performance:")
print(df_v2.mean(numeric_only=True))

V1 performance:
instructions_follow   NaN
answer_relevant       NaN
answer_clear          NaN
answer_citations      NaN
completeness          NaN
tool_call_search      NaN
answer_length         NaN
dtype: float64

V2 performance:
instructions_follow   NaN
answer_relevant       NaN
answer_clear          NaN
answer_citations      NaN
completeness          NaN
tool_call_search      NaN
answer_length         NaN
dtype: float64


In [157]:
print("User questions:")
print(df_user.mean(numeric_only=True))

print("\nAI-generated questions:")
print(df_ai.mean(numeric_only=True))

User questions:
instructions_follow      1.0
answer_relevant          1.0
answer_clear             1.0
answer_citations         0.9
completeness             0.9
tool_call_search         1.0
answer_length          972.8
dtype: float64

AI-generated questions:
instructions_follow       1.000000
answer_relevant           0.857143
answer_clear              1.000000
answer_citations          0.857143
completeness              0.857143
tool_call_search          1.000000
answer_length          1433.571429
dtype: float64


In [158]:
# --- REBUILD EVAL PIPELINE ---

eval_results = []
failed_logs = []

from pathlib import Path

log_files = list(LOG_DIR.glob("*.json"))

print("Total logs found:", len(log_files))

for log_file in log_files:
    try:
        log_record = load_log_file(log_file)
        eval_result = await evaluate_log_record(log_record)
        eval_results.append((log_record, eval_result))
    except Exception as e:
        failed_logs.append((log_file.name, str(e)))

print("Successful evals:", len(eval_results))
print("Failed evals:", len(failed_logs))


# --- BUILD DATAFRAME ---

import pandas as pd

rows = []

for log_record, eval_result in eval_results:
    messages = log_record["messages"]

    row = {
        "file": Path(log_record["log_file"]).name,
        "agent_name": log_record.get("agent_name"),
        "agent_version": log_record.get("agent_version"),
        "source": log_record.get("source"),
        "question": messages[0]["parts"][0]["content"],
        "answer": messages[-1]["parts"][0]["content"],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)

    rows.append(row)

df = pd.DataFrame(rows)

df.to_csv("evaluation_results.csv", index=False)

print("\n--- DISTRIBUTION ---")
print(df["source"].value_counts())
print(df["agent_version"].value_counts())

Total logs found: 18
Successful evals: 18
Failed evals: 0

--- DISTRIBUTION ---
source
user            10
ai-generated     8
Name: count, dtype: int64
agent_version
v3    18
Name: count, dtype: int64
